# MiniGPT on a faster engine: follow along

This notebook goes with my post [MiniGPT (Part 4)](https://haddley.github.io/posts/minigpt3/). It rebuilds Part 3's machine in Apple's MLX, checks that it is the same machine, trains it, races MLX against PyTorch, and writes a story.

**It needs a Mac with Apple Silicon**, because MLX only runs there. Open it in Jupyter from a clone of [minigpt-series](https://github.com/Haddley/minigpt-series), with the packages from its `requirements.txt` installed.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))
sys.path.insert(0, os.path.abspath("."))
from notebook_setup import setup
run = setup("part4-mlx")

## 1. The same machine, in two engines

Build Part 3's PyTorch machine and the MLX one with the same 8,192 token cards, and count their numbers.

In [ ]:
import importlib.util
import mlx.core as mx
from mlx.utils import tree_flatten
from model_mlx import MiniGPT as MLXMiniGPT

spec = importlib.util.spec_from_file_location("torch_model", "../part3-tokenisers/model.py")
torch_model = importlib.util.module_from_spec(spec); spec.loader.exec_module(torch_model)

pt = torch_model.MiniGPT(8192)
mlx_model = MLXMiniGPT(8192)
mx.eval(mlx_model.parameters())
print("PyTorch:", f"{sum(p.numel() for p in pt.parameters()):,}", "numbers")
print("MLX:    ", f"{sum(v.size for _, v in tree_flatten(mlx_model.parameters())):,}", "numbers")

## 2. Lazy evaluation

MLX writes calculations down and only runs them when a result is needed. Building a big multiplication is instant; asking for its result is where the work happens.

In [ ]:
import time
a = mx.random.normal((4096, 4096))
b = mx.random.normal((4096, 4096))
mx.eval(a, b)

t = time.time(); c = a @ b @ a @ b;  print(f"writing the steps down: {(time.time() - t) * 1000:8.2f} ms")
t = time.time(); mx.eval(c);         print(f"calculating them:       {(time.time() - t) * 1000:8.2f} ms")

## 3. Train the MLX machine

3,000 training steps on Part 3's stories, checking bits per byte every 300 steps. On my Mac Studio, 3,000 steps took about 8 minutes.

In [ ]:
run("train_mlx.py", "--tokenizer", "bpe8k", "--iters", "3000", "--eval-interval", "300")

## 4. The race

The same 200 timed training steps in each engine, each in its own fresh program, so the memory figures stay clean.

In [ ]:
for framework in ("torch", "mlx-nocompile", "mlx"):
    run("bench.py", "--framework", framework)

## 5. What it writes

In [ ]:
run("generate_mlx.py", "--prompt", "Once upon a time", "--max-new-tokens", "150")